In [1]:
from RRAM import *
from tqdm import tqdm

In [2]:
# Función que realiza el Montecarlo

# Parámetros iniciales:
espesor_dispositivo = 10        # nm
Atom_size = 0.25                 # nm

eje_x = round(espesor_dispositivo / Atom_size)
eje_y = round(espesor_dispositivo / Atom_size)

num_trampas = 600

# Defino la región que quiero privilegiar y su peso
regiones_pesos = [
    ((10, eje_x-10, eje_y-15, eje_y), 50),             # First three columns with higher weight
    ((10, eje_x-10, 0, 15), 50),                       # First three columns with higher weight
]

# Genero el grid inicial
actual_state = Generation.initial_state_priv(eje_x, eje_y, num_trampas, regiones_pesos)

total_simulation_time = 10
num_pasos = 10000
paso_temporal = total_simulation_time / num_pasos

voltaje_final = 3

# Configuraciones iniciales:
temperatura = 350
campo_electrico = 0
voltaje = 0
simulation_time = 0
corriente = 0

# Creo el vector de datos como una matriz de num_pasos filas y 7 columnas:
# (paso, tiempo simulacion, x, y, probabilidad recombinacion, velocidad, funcion a trozos)
num_datos = num_pasos*eje_x*eje_y
data = np.zeros((num_datos, 7))
re_index = 0

In [3]:
# Comienza el algoritmo de Montecarlo
for k in tqdm(range(1, num_pasos+1)):

    # Guardo el estado anterior
    last_state = actual_state

    simulation_time = paso_temporal*k

    # Calculo la corriente
    voltaje += voltaje_final * paso_temporal

    # Obtengo la corrriente, antes decido cual usar comprobando si ha percolado o no
    # TODO: Revisar la corriente óhmica que no funciona
    if Percolation.is_path(actual_state):
        # Si ha percolado uso la corriente de percolación
        # Corriente = CurentSolver.OmhCurrent(Temperatura, Campo_Electrico)
        print("Ha percolado")
        break
    else:
        # Si no ha percolado uso la corriente de campo
        # TODO: REVISAR QUE LA CORRIENTE TIENE LAS UNIDADES CORRECTAS PORQUE NO CUADRAN VALORES cuando coloco la superficie
        corriente = CurentSolver.poole_frenkel(temperatura, campo_electrico)

    # Obtengo los valores del campo eléctrico y la temperatura
    campo_electrico = SimpleElectricField(voltaje, espesor_dispositivo*1e-9)
    # Temperatura = Temperature_Joule(voltaje, Corriente, T_0=350)

    # Calculo la probabilidad de generación o recombinación para ello recorro toda la matriz
    for i in range(eje_x):
        for j in range(eje_y):
            if actual_state[i, j] == 0:
                # TODO: REVISAR PROBABILIDAD QUE A VECES SALE MAYOR DE 1
                # TODO: HACER UN REESCALADO DE LOS VALORES PARA EVITAR TENER QUE TRABAJAR CON NUMEROS TAN GRANDES
                prob_generacion = Generation.generation(paso_temporal, campo_electrico, temperatura)
                random_number = np.random.rand()
                if random_number < prob_generacion:
                    actual_state[i, j] = 1  # Generación

            if actual_state[i, j] == 1:
                # TODO: REVISAR PROBABILIDAD QUE A VECES SALE MAYOR DE 1
                # TODO: HACER UN REESCALADO DE LOS VALORES PARA EVITAR TENER QUE TRABAJAR CON NUMEROS TAN GRANDES
                prob_rec, espacio_recorr, funcion_trozos = Recombination.recombination(
                    paso_temporal, i, campo_electrico, temperatura, Atom_size, factor=1000)
                data[re_index] = np.array([k, simulation_time, i, j, prob_rec, espacio_recorr, funcion_trozos])
                re_index += 1

                # genero un número aleatorio entre 0 y 1
                random_number = np.random.rand()
                if random_number < prob_rec:
                    actual_state[i, j] = 0  # Recombinación

 18%|█▊        | 1848/10000 [00:36<02:39, 51.06it/s] 

Ha percolado


In [4]:
# Guardo los datos generados

# Suponiendo que 'data' es un array de NumPy que ya contiene tus datos
data_filtrados = np.array([fila for fila in data if fila[-1] != 0.0])

# Creo el archivo para guardar losd datos
np.savetxt('Recombinacion_data.csv', data_filtrados,  fmt=['%d', '%.5f', '%d', '%d', '%f', '%.6e', '%.1f'],
           header='paso, tiempo simulacion, x, y, probabilidad recombinacion, velocidad, funcion a trozos', comments='', delimiter=', ')